# A Simple Diffusion Model on the MNIST Data Set using JAX / FLAX

This notebook is intended to store a simple implementation of a [diffusion model](https://en.wikipedia.org/wiki/Diffusion_model) as well as its training process on the [MNIST data set](https://www.tensorflow.org/datasets/catalog/mnist) using the Python libraries [JAX](https://docs.jax.dev/en/latest/index.html) and [FLAX](https://flax.readthedocs.io/en/stable/).

<center>
    <img src=imgs/jax.webp>
</center>

Let us first import the necessary data set. In this project, we are going to use the $(28 \times 28)$ MNIST images that are contained in the `tensorflow` package.

First of all, we are going to import the necessary modules.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds

2026-01-23 15:53:25.741180: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-23 15:53:28.612401: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769180008.803370 1598759 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769180008.872653 1598759 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769180010.487632 1598759 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

## Loading the MNIST Data

This section deals with loading the MNIST data set from the `tensorflow_datasets` package.

In [9]:
# Load the actual data
with tf.device("/cpu:0"):
    train_ds = tfds.load(name="mnist", split="train")
    X = np.array([batch["image"] for batch in train_ds.as_numpy_iterator()]).astype(np.float16)
    y = np.array([batch["label"] for batch in train_ds.as_numpy_iterator()]).astype(np.int8)

# Normalize to interval [-1, 1]
X = -1 + X * 2 / 255.0

# Resize from (28 x 28) to (32 x 32)
b, _, _, c = X.shape
X = tf.image.resize(X, size=(32, 32), method="nearest").numpy()

X.shape, y.shape, X.max(), X.min()

((60000, 32, 32, 1), (60000,), np.float16(1.0), np.float16(-1.0))

We have resized the images from $(28 \times 28)$ pixels to $(32 \times 32)$ pixels in order to make the downsampling steps more convenient later.

Let us display some of the images.

In [10]:
import ipywidgets

def plot(index):
    image = X[index]
    plt.imshow(image, cmap="gray")
    plt.title(f"Image of class {y[index]}")

ipywidgets.interact(plot, index=ipywidgets.IntSlider(min=0, max=X.shape[0], step=1))

interactive(children=(IntSlider(value=0, description='index', max=60000), Output()), _dom_classes=('widget-int…

<function __main__.plot(index)>

## Implementing the Diffusion Model

**TODO**